In [ ]:
!git clone https://github.com/Vinuitik/TypiClustImplement.git
%cd TypiClustImplement

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch

_ROOT = Path.cwd()
_DEEPAL_DIR = _ROOT / 'deep-active-learning'
if str(_DEEPAL_DIR) not in sys.path:
    sys.path.insert(0, str(_DEEPAL_DIR))

from data import get_CIFAR10
from handlers import CIFAR10_Handler
from utils import get_net

from al_methods import STRATEGY_REGISTRY, select_indices, typiclust, typiclust_improv

BUDGETS = [10, 20, 30, 40, 50, 60, 100, 150, 200, 250, 300]
METHODS = sorted(STRATEGY_REGISTRY.keys()) + ['typiclust', 'typiclust_improv']
EMBEDDINGS_PATH = str(_ROOT / 'datasets' / 'cifar10_train_embeddings.npz')
N_CLASSES = 10  # CIFAR-10
OUTPUT_DIR = _ROOT / 'datasets' / 'al_splits'
SEED = 42

print('CUDA available:', torch.cuda.is_available())
print('Methods:', METHODS)

In [ ]:
def _sorted_ints(values):
    return sorted(int(v) for v in values)


def _run_method(method: str) -> None:
    budgets = sorted(BUDGETS)

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    dataset = get_CIFAR10(CIFAR10_Handler)

    if method == 'typiclust':
        # Restrict pool to match embeddings size (embeddings may cover fewer samples than dataset)
        n_emb = int(np.load(EMBEDDINGS_PATH)['embeddings'].shape[0])
        dataset.X_train = dataset.X_train[:n_emb]
        dataset.Y_train = dataset.Y_train[:n_emb]
        dataset.n_pool = n_emb
        dataset.labeled_idxs = dataset.labeled_idxs[:n_emb]
    else:
        # Stratified init: 1 sample per class — guarantees all classes present
        # (DeepAL strategies need a trainable labeled set before the first query)
        y = dataset.Y_train.numpy()
        for cls in range(N_CLASSES):
            cls_idxs = np.where(y == cls)[0]
            chosen = np.random.choice(cls_idxs, 1, replace=False)
            dataset.labeled_idxs[chosen] = True

    labeled = np.where(dataset.labeled_idxs)[0].tolist()
    unlabeled = np.where(~dataset.labeled_idxs)[0].tolist()

    net = None
    if method in STRATEGY_REGISTRY and method != 'random':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        net = get_net('CIFAR10', device)

    current_labeled_count = len(labeled)

    for budget in budgets:
        increment = budget - current_labeled_count

        if increment > 0:
            if method in STRATEGY_REGISTRY:
                selected, unlabeled = select_indices(
                    dataset=dataset,
                    budget=increment,
                    strategy=method,
                    net=net,
                    train_before_query=(net is not None),
                    update_dataset_labels=True,
                )
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            elif method == 'typiclust':
                selected, unlabeled = typiclust(
                    dataset=dataset,
                    budget=budget,
                    embeddings_npz_path=EMBEDDINGS_PATH,
                )
                for idx in selected:
                    dataset.labeled_idxs[idx] = True
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            current_labeled_count = len(labeled)

        out_path = OUTPUT_DIR / f'{method}_{budget}.json'
        out_path.write_text(
            json.dumps({
                'labeled_indices': _sorted_ints(labeled),
                'unlabeled_indices': _sorted_ints(unlabeled),
            }),
            encoding='utf-8',
        )
        print(f'  saved {out_path.name}  ({len(labeled)} labeled)')

In [ ]:
def _sorted_ints(values):
    return sorted(int(v) for v in values)


def _run_method(method: str) -> None:
    budgets = sorted(BUDGETS)

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    dataset = get_CIFAR10(CIFAR10_Handler)

    if method in ('typiclust', 'typiclust_improv'):
        # embedding-based methods — can select from a fully unlabeled pool
        pass
    else:
        # Stratified init: 1 sample per class — guarantees all classes present
        # (DeepAL strategies need a trainable labeled set before the first query)
        y = dataset.Y_train.numpy()
        for cls in range(N_CLASSES):
            cls_idxs = np.where(y == cls)[0]
            chosen = np.random.choice(cls_idxs, 1, replace=False)
            dataset.labeled_idxs[chosen] = True

    labeled = np.where(dataset.labeled_idxs)[0].tolist()
    unlabeled = np.where(~dataset.labeled_idxs)[0].tolist()

    net = None
    if method in STRATEGY_REGISTRY and method != 'random':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        net = get_net('CIFAR10', device)

    current_labeled_count = len(labeled)

    for budget in budgets:
        increment = budget - current_labeled_count

        if increment > 0:
            if method in STRATEGY_REGISTRY:
                selected, unlabeled = select_indices(
                    dataset=dataset,
                    budget=increment,
                    strategy=method,
                    net=net,
                    train_before_query=(net is not None),
                    update_dataset_labels=True,
                )
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            elif method == 'typiclust':
                selected, unlabeled = typiclust(
                    dataset=dataset,
                    budget=budget,
                    embeddings_npz_path=EMBEDDINGS_PATH,
                )
                for idx in selected:
                    dataset.labeled_idxs[idx] = True
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            elif method == 'typiclust_improv':
                selected, unlabeled = typiclust_improv(
                    dataset=dataset,
                    budget=budget,
                    embeddings_npz_path=EMBEDDINGS_PATH,
                )
                for idx in selected:
                    dataset.labeled_idxs[idx] = True
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            current_labeled_count = len(labeled)

        out_path = OUTPUT_DIR / f'{method}_{budget}.json'
        out_path.write_text(
            json.dumps({
                'labeled_indices': _sorted_ints(labeled),
                'unlabeled_indices': _sorted_ints(unlabeled),
            }),
            encoding='utf-8',
        )
        print(f'  saved {out_path.name}  ({len(labeled)} labeled)')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for method in METHODS:
    if method == 'adversarial_bim' or method == 'adversarial_deepfool':
        continue
    print(f'[{method}]')
    _run_method(method)
print('Done.')

In [ ]:
# Zip all generated splits for easy download
import shutil
shutil.make_archive('al_splits', 'zip', OUTPUT_DIR)
print('al_splits.zip ready for download')